# Домашнее задание 1. Первая модель на данных о покупках

## Дедлайн: до 23:59 7 декабря 2026 года

### Заполненный jupyter-ноутбук необходимо отправить на почту aaasenin@gmail.com
### В теме письма указать "ДЗ_1_<ваше ФИО>", в названии файла ноутбука указать ваше ФИО

**Максимум — 11 баллов.** Повторяем путь из практики: загрузить таблицы, выбрать данные,
сложить покупки по клиентам, присоединить target и обучить модель.

Можно пользоваться практиками и подсказками под заданиями, копировать и адаптировать код.
В заготовках замените `...` нужным именем файла, столбца или переменной.
Строки без пропусков уже готовы — их достаточно запустить.

Баллы начисляются за выполненные шаги. **Высокая accuracy или recall для получения 11 баллов не требуется.**
Подбирать параметры модели, строить графики и улучшать прогноз не нужно.

| № | Задание | Баллы |
|---|---|---:|
| 1 | Загрузить две таблицы | 1 |
| 2 | Выбрать столбцы | 1 |
| 3 | Оставить покупки мая | 1 |
| 4 | Посчитать сумму покупок каждого клиента | 2 |
| 5 | Перевести сумму в тысячи рублей | 1 |
| 6 | Соединить признаки и target | 1 |
| 7 | Подготовить X, y и разделить клиентов | 1 |
| 8 | Выполнить fit и predict | 2 |
| 9 | Посчитать accuracy и recall | 1 |
| | **Итого** | **11** |

## Файлы и обозначения

Положите `homework_purchases.csv` и `homework_target.csv` рядом с ноутбуком.
В Colab загрузите их в панель «Файлы». Выполняйте задания по порядку.

В `homework_purchases.csv` **одна строка — одна покупка**:

| Столбец | Смысл |
|---|---|
| `operation_id` | ID покупки |
| `client_id` | ID клиента |
| `month` | Месяц: `2026-04` или `2026-05` |
| `channel` | Канал покупки: «Карта» или «Онлайн» |
| `amount_rub` | Сумма покупки в рублях |

В `homework_target.csv` **одна строка — один клиент**:

- `client_id` — ID клиента.
- `target = 1` — в июне покупок не было; `target = 0` — в июне покупки были.

По покупкам **мая** предсказываем исход **июня**. Покупки мая есть у всех 40 клиентов из target.
Пропусков и повторных операций в файлах нет: дополнительная очистка не требуется.

Запустите ячейку с готовыми импортами:

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score

## Задание 1. Загрузите таблицы — 1 балл

Прочитайте оба CSV в переменные `purchases` и `target`. Посмотрите первые строки.

**Подсказка:** в первую строку подставьте `homework_purchases.csv`, во вторую — `homework_target.csv`.
Названия файлов пишутся в кавычках.

**Баллы:** по 0,5 за каждую загруженную таблицу.

In [ ]:
purchases = pd.read_csv("...")
target = pd.read_csv("...")
purchases.head()

In [ ]:
target.head()

## Задание 2. Выберите столбцы — 1 балл

Покажите первые пять строк только со столбцами `client_id`, `month` и `amount_rub`.
Исходную таблицу менять не нужно.

**Подсказка:** для нескольких столбцов нужны две пары квадратных скобок.
Вместо двух пропусков подставьте `month` и `amount_rub`.

In [ ]:
purchases[["client_id", "...", "..."]].head()

## Задание 3. Оставьте только май — 1 балл

Сохраните в `may` покупки за `2026-05`. Апрельские покупки туда попасть не должны.

**Подсказка:** справа от `==` подставьте месяц `2026-05`. Дату преобразовывать не нужно:
здесь сравниваем два текста. `.copy()` создаёт отдельную таблицу отобранных строк.

In [ ]:
may = purchases[purchases["month"] == "..."].copy()
may.head()

## Задание 4. Сложите покупки каждого клиента — 2 балла

Сначала соберите майские покупки одного клиента вместе и сложите их суммы.
Сохраните результат в `total_by_client`.

**Подсказка:** группировать нужно по `client_id`. Всё остальное в команде уже написано.
Одна группа — все майские покупки одного клиента.

**Баллы:** 1 за суммы по клиентам, 1 за таблицу `features` в следующей ячейке.

In [ ]:
total_by_client = may.groupby("...")["amount_rub"].sum()
total_by_client.head()

Теперь верните ID клиента в обычный столбец. Эта ячейка готова — запустите её.
Должна получиться таблица из **40 строк** с колонками `client_id` и `amount_rub`.

In [ ]:
features = total_by_client.reset_index()
features.head()

## Задание 5. Создайте признак — 1 балл

Добавьте в `features` столбец `spending_thousand_rub`: сумма покупок в тысячах рублей.
Например, 2 500 ₽ превратятся в 2,5 тыс. ₽.

**Подсказка:** разделите столбец `amount_rub` на 1000. Подставить нужно только имя столбца.

In [ ]:
features["spending_thousand_rub"] = features["..."] / 1000
features.head()

## Задание 6. Присоедините признаки к target — 1 балл

Соедините `target` и `features` в таблицу `data`.
У каждого клиента должны оказаться рядом правильный ответ и его сумма покупок.

**Подсказка:** общий столбец в обеих таблицах — `client_id`. Подставьте его в `on`.
`how="left"` сохраняет всех клиентов из таблицы `target`.

In [ ]:
data = target.merge(features, on="...", how="left")
data.head()

## Задание 7. Подготовьте X, y и разделите клиентов — 1 балл

В **X** возьмите один признак `spending_thousand_rub`. В **y** — правильный ответ `target`.
ID клиента и сам target в X не включаем.

**Подсказка:** в первой строке подставьте `spending_thousand_rub`, во второй — `target`.
Обратите внимание: у X две пары квадратных скобок, у y — одна.

**Баллы:** 0,5 за X и y; 0,5 за выполненное разделение в следующей готовой ячейке.

In [ ]:
X = data[["..."]]
y = data["..."]

Следующую ячейку менять не нужно. Она оставляет 30 клиентов для обучения и 10 для проверки.
`stratify=y` помогает сохранить соотношение классов, `random_state=42` — повторить то же разделение.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Клиентов для обучения:", len(X_train))
print("Клиентов для проверки:", len(X_test))

## Задание 8. Обучите модель и получите прогноз — 2 балла

**Сначала fit:** передайте модели признаки и ответы клиентов для обучения.

**Подсказка:** в `fit` подставьте **`X_train, y_train`** именно в таком порядке.

**Баллы:** 1 за fit, 1 за predict.

In [ ]:
model = LogisticRegression()
model.fit(..., ...)
print("Модель обучена")

**Теперь predict:** получите прогнозы для клиентов проверки.

**Подсказка:** в `predict` передаём только **`X_test`**, без правильных ответов.

In [ ]:
predictions = model.predict(...)
predictions

## Задание 9. Посчитайте две метрики — 1 балл

Вычислите accuracy и recall на клиентах проверки.

- Accuracy — доля правильных прогнозов среди всех клиентов проверки.
- Recall — доля найденных клиентов с фактом `target = 1` среди всех клиентов с фактом 1.

**Подсказка:** обе функции получают одни и те же аргументы: сначала **`y_test`**, затем **`predictions`**.
Правильный вызов устроен так: `имя_метрики(y_test, predictions)`.

**Баллы:** по 0,5 за каждую рассчитанную метрику. Требования к её величине нет.

In [ ]:
accuracy = accuracy_score(..., ...)
recall = recall_score(..., ..., zero_division=0)

print("Accuracy:", round(accuracy, 2))
print("Recall:", round(recall, 2))

Эту готовую ячейку можно запустить, чтобы увидеть, какие прогнозы совпали с фактами:

In [ ]:
pd.DataFrame({"Факт": y_test, "Прогноз": predictions})

## Что сдать

Сохраните этот `.ipynb` со своим кодом и результатами выполнения. В нём должны быть видны
таблица после объединения, сообщение об обучении, прогнозы и две метрики.

Отдельный отчёт и дополнительные файлы создавать не нужно.